# 06 - Library-size sweep: confirmatory runs (reduced design, PROTOCOL_sweep.md section 5)

Runs `configs/sweep_reduced.yaml` in the registered order: Qwen2.5-1.5B seeds 0-9, then Qwen2.5-3B seeds 0-7, every arm.
One Kaggle session cannot hold the whole design, so the notebook is re-run until nothing is left:

- it starts from `results/sweep_reduced/records.jsonl` in the attached repo (every earlier session's rows, merged), so arms
  already marked done are skipped;
- it starts no new (model, seed) after `MAX_HOURS`; a seed cut off by the session limit loses only its unfinished arm,
  whose rows are dropped by the analysis (`finished_runs_only`);
- its output `runs/sweep_reduced/records.jsonl` is merged back into the repo by `scripts/kaggle_sweep.py`.

## Setup

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = ""  # optional: a git URL for this repo; leave empty to use an attached dataset
WORK = "/kaggle/working/skill-library"

if not os.path.exists(WORK):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
    else:
        found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/PROTOCOL_sweep.md", recursive=True)]
        if not found:
            raise SystemExit("attach the repo as a dataset or set REPO_URL")
        shutil.copytree(found[0], WORK)
sys.path.insert(0, WORK)
os.chdir(WORK)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft>=0.11", "transformers>=4.40", "accelerate", "pyyaml"], check=False)
# Kaggle images ship an old torchao that recent peft rejects at LoRA injection (ImportError); nothing here uses torchao.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "this notebook needs a GPU accelerator"
GPU = torch.cuda.get_device_name(0)
print(GPU, f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB", "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
import shutil
from pathlib import Path
from sweep.config import load_sweep_config
from sweep.runner import finished_arms

cfg = load_sweep_config("configs/sweep_reduced.yaml")
OUT = Path(cfg.out_dir)
RECORDS = OUT / "records.jsonl"
OUT.mkdir(parents=True, exist_ok=True)
prior = Path("results/sweep_reduced/records.jsonl")
if prior.exists() and not RECORDS.exists():
    shutil.copyfile(prior, RECORDS)
arm_keys = {a.key for a in cfg.arm_list()}
TODO = [(m, s) for m in cfg.models for s in cfg.seeds_for(m) if not arm_keys <= finished_arms(RECORDS, m, s)]
print(len(TODO), "(model, seed) pairs left:", TODO)

## Run (resumes; stops starting seeds after MAX_HOURS)

In [ ]:
import gc, time
from sweep.backends import HFSweepBackend
from sweep.runner import run_model_seed, write_manifest

MAX_HOURS = 8.5          # no new (model, seed) after this; Kaggle stops GPU sessions at 12 h
started, backend, loaded = time.time(), None, None
for model, seed in TODO:
    if (time.time() - started) / 3600 > MAX_HOURS:
        print("time budget reached; the next session continues from here")
        break
    if model != loaded:
        backend = None
        gc.collect(); torch.cuda.empty_cache()
        backend, loaded = HFSweepBackend(cfg, model, seed=seed), model
        write_manifest(cfg, OUT, {"gpu": GPU, "notebook": "06_sweep_confirmatory"})
    backend.seed = seed
    run_model_seed(cfg, model, seed, backend, RECORDS)
    torch.cuda.empty_cache()
left = [(m, s) for m in cfg.models for s in cfg.seeds_for(m) if not arm_keys <= finished_arms(RECORDS, m, s)]
print("left after this session:", left)